# 03 — Train POLO

Train the deployed **polo26n** model on the feeder-only dataset with the locked
hyperparameters from Johan's W&B sweep, plus optional reproducibility helpers.

- **Input:** `feeder_only/data.yaml` from `02_annotation_prep`
- **Output:** trained weights under `models/polo-yolo11/<run_name>/weights/best.pt`

> The deployed model is `polo26n`. Definitive evaluation numbers come from
> `04_evaluation` (class-agnostic Hungarian matching), not from the POLO
> built-in validation called here (which uses DoR matching as a quick check).

In [1]:
import sys
from datetime import datetime
from pathlib import Path

from mosaic.core.dataset import Dataset
import mosaic.tracking.pose_training as pose

# Make the repo-root config.py importable regardless of the kernel's working dir
_REPO_ROOT = Path.cwd()
while not (_REPO_ROOT / 'config.py').exists() and _REPO_ROOT != _REPO_ROOT.parent:
    _REPO_ROOT = _REPO_ROOT.parent
sys.path.insert(0, str(_REPO_ROOT))
import config

## Configuration

In [2]:
DATASET_BASE = Path('/Volumes/JD-SSD/feeder-detection-model/dataset')          # local smoke-test snapshot
# DATASET_BASE = Path('/home/beesbook/feeder_detection_model/dataset')  # live dataset on the training machine
ds = Dataset(DATASET_BASE / 'dataset.yaml').load()

DATA_YAML = config.ensure_absolute_data_yaml(config.feeder_only_dir(ds) / 'data.yaml')
OUTPUT_DIR = ds.get_root('models') / config.POLO_MODEL_NAME
DEVICE = config.auto_device()

print('data.yaml:', DATA_YAML)
print('device:', DEVICE)
pose.check_dataset(str(DATA_YAML.parent))

data.yaml: /Volumes/JD-SSD/feeder-detection-model/dataset/models/polo/_polo_data/feeder_only/data.yaml
device: mps
Dataset: /Volumes/JD-SSD/feeder-detection-model/dataset/models/polo/_polo_data/feeder_only

[train] labels: /Volumes/JD-SSD/feeder-detection-model/dataset/models/polo/_polo_data/feeder_only/train/labels
  Class counts: {0: 5968, 1: 346, 3: 173}
  Empty: 0 / 977

[valid] labels: /Volumes/JD-SSD/feeder-detection-model/dataset/models/polo/_polo_data/feeder_only/valid/labels
  Class counts: {0: 932, 1: 23}
  Empty: 0 / 133

[test] labels: /Volumes/JD-SSD/feeder-detection-model/dataset/models/polo/_polo_data/feeder_only/test/labels
  Class counts: {0: 992, 1: 29}
  Empty: 0 / 136


## Train the deployed model (polo26n, locked hyperparameters)

`config.POLO_FINAL` holds the locked configuration. **NOTE:** the model config
string is `polo26n.yaml` (`config.POLO_MODEL_CFG`); an earlier POLO build used
`polov8n.yaml`. If training errors on an unknown model config, switch it in
`config.py` to match the installed POLO fork.

In [ ]:
run_name = 'polo26n_' + datetime.now().strftime('%Y%m%d_%H%M%S')
hp = dict(config.POLO_FINAL)
print('Hyperparameters:', hp)

results = pose.train_point_model(
    data_yaml=DATA_YAML,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name=run_name,
    **hp,
)
best = pose.find_best_model(str(OUTPUT_DIR))
print('Best model:', best)

In [ ]:
# Quick built-in (DoR) validation on the test split — sanity check only.
pose.validate_point_model(
    model_path=best,
    data_yaml=DATA_YAML,
    device=DEVICE,
    imgsz=config.IMGSZ,
    dor=config.DOR,
    split='test',
)

## Optional — selected retrain of polo26n / s / m (reproducibility)

Retrains all three architectures with the same locked hyperparameters so
`04_evaluation` can confirm they reproduce the published model-size table. Cheap
relative to the full Bayesian sweep.

In [ ]:
RETRAIN_ALL_SIZES = False  # set True to retrain polo26n / polo26s / polo26m

if RETRAIN_ALL_SIZES:
    for name, model_cfg in config.POLO_SWEEP_MODELS.items():
        hp = dict(config.POLO_FINAL)
        hp['model'] = model_cfg
        rn = f'{name}_' + datetime.now().strftime('%Y%m%d_%H%M%S')
        print(f'Training {name} ({model_cfg}) -> {rn}')
        pose.train_point_model(
            data_yaml=DATA_YAML, device=DEVICE,
            project=str(OUTPUT_DIR), name=rn, **hp,
        )
else:
    print('Skipping multi-size retrain (RETRAIN_ALL_SIZES=False)')

## Reference — W&B hyperparameter sweep

The locked hyperparameters above were selected by a Bayesian W&B sweep over
`polo26s` / `polo26m`. The configuration is recorded here for provenance (run it
with the W&B sweep agent; not required to train the deployed model). See
`docs/model-comparison.md` for results.

In [ ]:
SWEEP_CONFIG = {
    'method': 'bayes',
    'metric': {'name': 'val/f1', 'goal': 'maximize'},
    'parameters': {
        'model':        {'values': ['polo26s.yaml', 'polo26m.yaml']},
        'loc':          {'min': 3.0, 'max': 7.0},
        'lr0':          {'min': 1e-3, 'max': 3e-2, 'distribution': 'log_uniform_values'},
        'lrf':          {'min': 1e-3, 'max': 2e-2, 'distribution': 'log_uniform_values'},
        'weight_decay': {'min': 1e-4, 'max': 2e-3, 'distribution': 'log_uniform_values'},
        'augmentation': {'values': ['medium', 'heavy']},
    },
}
SWEEP_FIXED = {'epochs': 200, 'imgsz': 640, 'batch': 8, 'dor': 0.3,
               'patience': 50, 'loc_loss': 'mse'}
# NB: the sweep logged validation F1 at DoR=0.8 (depressing F1 to ~0.85);
# deployment + the definitive eval use DoR=0.3.
SWEEP_CONFIG

## Localizer baseline (for the old-vs-POLO comparison)

Trains the legacy 2019 BeesBook localizer on the feeder patches. Only needed to
populate the old-vs-POLO row in `04_evaluation`; the patch dataset must be
prepared separately (128x128 grayscale patches at the localizer scale).

In [ ]:
from mosaic.tracking.pose_training import LocalizerAugmentConfig

TRAIN_LOCALIZER = False
if TRAIN_LOCALIZER:
    loc_dataset_dir = ds.get_root('models') / 'beesbook-localizer' / '_localizer_data' / 'cvat'
    result = pose.train_localizer(
        dataset_dir=loc_dataset_dir,
        num_classes=config.NUM_CLASSES,
        initial_channels=config.INITIAL_CHANNELS,
        weights=None,  # or path to the 2019 pretrained .h5 / .pt
        epochs=config.LOCALIZER_DEFAULTS['epochs'],
        batch_size=config.LOCALIZER_DEFAULTS['batch_size'],
        lr=config.LOCALIZER_DEFAULTS['lr'],
        early_stopping_patience=config.LOCALIZER_DEFAULTS['early_stopping_patience'],
        lr_patience=config.LOCALIZER_DEFAULTS['lr_patience'],
        device=DEVICE,
        project=str(ds.get_root('models') / 'beesbook-localizer'),
        name='finetune',
        seed=42,
        augment=LocalizerAugmentConfig(flip_h=True, flip_v=True, rotate_90=True),
    )
    print('Localizer best:', result.best_model_path)
else:
    print('Skipping localizer training (TRAIN_LOCALIZER=False)')